### RAG pipeline


### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
#from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path
import re
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import xml.etree.ElementTree as ET

from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_13940\377539602.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:

def process_all_documents(input_directory):
    """
    Read all PDF and XML files recursively from a directory.

    Returns:
        List of LangChain Document objects.
    """

    all_documents = []

    input_dir = Path(input_directory)

    if not input_dir.exists():
        raise FileNotFoundError(
            f"Input directory does not exist: {input_dir.resolve()}"
        )

    # Find all PDF and XML files recursively
    supported_files = [
        file for file in input_dir.rglob("*")
        if file.is_file() and file.suffix.lower() in [".pdf", ".xml"]
    ]

    print(f"Found {len(supported_files)} supported files")

    for file_path in supported_files:

        print(f"\nProcessing: {file_path.name}")

        try:

            # --------------------------------
            # 1. Read PDF files
            # --------------------------------

            if file_path.suffix.lower() == ".pdf":

                loader = PyPDFLoader(str(file_path))
                documents = loader.load()

                for doc in documents:
                    doc.metadata["source_file"] = file_path.name
                    doc.metadata["file_type"] = "pdf"

            # --------------------------------
            # 2. Read XML files
            # --------------------------------

            elif file_path.suffix.lower() == ".xml":

                tree = ET.parse(file_path)
                root = tree.getroot()

                # Extract XML text while retaining element hierarchy
                def extract_xml_text(element, level=0):

                    content = []

                    tag = element.tag.split("}")[-1]

                    text = (element.text or "").strip()

                    if text:
                        content.append(
                            "  " * level + f"{tag}: {text}"
                        )

                    for child in element:
                        content.extend(
                            extract_xml_text(child, level + 1)
                        )

                    return content

                xml_text = "\n".join(
                    extract_xml_text(root)
                )

                documents = [
                    Document(
                        page_content=xml_text,
                        metadata={
                            "source_file": file_path.name,
                            "file_type": "xml"
                        }
                    )
                ]

            # --------------------------------
            # 3. Add documents to main list
            # --------------------------------

            all_documents.extend(documents)

            print(f"Loaded {len(documents)} document(s)")

        except Exception as e:

            print(f"Error processing {file_path.name}: {e}")

    print("\n-----------------------------")
    print(f"Total documents loaded: {len(all_documents)}")
    print("-----------------------------")

    return all_documents


In [3]:
all_documents = process_all_documents(r"E:\RAG\RAG-test\Input Files")

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)
Ignoring wrong pointing object 58 0 (offset 0)
Ignoring wrong pointing object 60 0 (offset 0)
Ignoring wrong pointing object 62 0 (offset 0)
Ignoring wrong pointing object 70 0 (offset 0)
Ignoring wrong pointing object 72 0 (offset 0)
Ignoring wrong pointing object 82 0 (offset 0)
Ignoring wrong pointing object 91 0 (offset 0)
Ignoring wrong pointing object 174 0 (offset 0)


Found 5 supported files

Processing: 16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf


Ignoring wrong pointing object 180 0 (offset 0)
Ignoring wrong pointing object 187 0 (offset 0)
Ignoring wrong pointing object 196 0 (offset 0)
Ignoring wrong pointing object 231 0 (offset 0)
Ignoring wrong pointing object 244 0 (offset 0)
Ignoring wrong pointing object 252 0 (offset 0)
Ignoring wrong pointing object 254 0 (offset 0)
Ignoring wrong pointing object 267 0 (offset 0)
Ignoring wrong pointing object 274 0 (offset 0)


Loaded 37 document(s)

Processing: Global and local indicators of spatial connectivity for.pdf
Loaded 34 document(s)

Processing: Saturn paper.pdf
Loaded 43 document(s)

Processing: Attention Is All You Need.xml
Loaded 1 document(s)

Processing: Generative Agents Interactive Simulacra of Human Behavior.xml
Loaded 1 document(s)

-----------------------------
Total documents loaded: 116
-----------------------------


In [4]:

def split_documents(
    documents,
    chunk_size=1500,
    chunk_overlap=200
):
    """
    Generic hierarchical document splitter.

    Detects:
        - Chapters
        - Sections
        - Subsections
        - Lower-level headings

    Preserves:
        - Original source metadata
        - Document name and file type
        - Hierarchical heading information
        - Page information
    """

    # ---------------------------------------
    # 1. Text splitter
    # ---------------------------------------

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=[
            "\n\n",
            "\n",
            ". ",
            " ",
            ""
        ]
    )

    # ---------------------------------------
    # CHANGE 1: Correct heading regex patterns
    # ---------------------------------------

    # Matches: 1 Introduction, 2.1 Background,
    # 2.1.1 Research objectives
    numbered_heading = re.compile(
        r"^\s*(\d+(?:\.\d+)*\.?)\s+(.+?)\s*$"
    )

    # Matches: Chapter 1 Introduction, Part II Overview
    chapter_heading = re.compile(
        r"^\s*(chapter|part)\s+"
        r"([0-9ivxlcdm]+)\b[\s:.-]*(.*?)\s*$",
        re.IGNORECASE
    )

    # Matches: # Chapter, ## Section, ### Subsection
    markdown_heading = re.compile(
        r"^\s*(#{1,6})\s+(.+?)\s*$"
    )

    structured_chunks = []

    # ---------------------------------------
    # CHANGE 2: Expanded hierarchy metadata
    # ---------------------------------------

    hierarchy = {
        "chapter_number": None,
        "chapter_title": None,
        "section_number": None,
        "section_title": None,
        "subsection_number": None,
        "subsection_title": None,
        "heading": None
    }

    buffer = []
    page_numbers = set()

    source_metadata = {}
    current_source = None

    # ---------------------------------------
    # CHANGE 3: Preserve source metadata
    # ---------------------------------------

    def flush_buffer():

        nonlocal buffer, page_numbers

        text = "\n".join(buffer).strip()

        if not text:
            return

        chunks = splitter.split_text(text)

        # Add hierarchy to the chunk text itself.
        # This helps semantic retrieval understand context.
        heading_context = " > ".join(
            str(value)
            for value in [
                hierarchy["chapter_title"],
                hierarchy["section_title"],
                hierarchy["subsection_title"],
                hierarchy["heading"]
            ]
            if value
        )

        for index, chunk in enumerate(chunks):

            metadata = {
                **source_metadata,
                **hierarchy,
                "pages": ", ".join(
                    map(str, sorted(page_numbers))
                ),
                "chunk_index": index
            }

            # Remove None values and unsupported metadata
            metadata = {
                key: value
                for key, value in metadata.items()
                if value is not None
                and isinstance(value, (str, int, float, bool))
            }

            # Include heading context in retrieved content
            content = (
                f"Document hierarchy: {heading_context}\n\n"
                f"{chunk}"
                if heading_context
                else chunk
            )

            structured_chunks.append(
                Document(
                    page_content=content,
                    metadata=metadata
                )
            )

        buffer = []
        page_numbers = set()

    # ---------------------------------------
    # CHANGE 4: Update hierarchy dynamically
    # ---------------------------------------

    def update_hierarchy(level, number, title):

        levels = [
            "chapter",
            "section",
            "subsection",
            "heading"
        ]

        position = levels.index(level)

        # Clear lower levels
        for lower_level in levels[position:]:
            if lower_level == "chapter":
                hierarchy["chapter_number"] = None
                hierarchy["chapter_title"] = None

            elif lower_level == "section":
                hierarchy["section_number"] = None
                hierarchy["section_title"] = None

            elif lower_level == "subsection":
                hierarchy["subsection_number"] = None
                hierarchy["subsection_title"] = None

            elif lower_level == "heading":
                hierarchy["heading"] = None

        # Assign the new heading
        if level == "chapter":
            hierarchy["chapter_number"] = number
            hierarchy["chapter_title"] = title

        elif level == "section":
            hierarchy["section_number"] = number
            hierarchy["section_title"] = title

        elif level == "subsection":
            hierarchy["subsection_number"] = number
            hierarchy["subsection_title"] = title

        elif level == "heading":
            hierarchy["heading"] = title

    # ---------------------------------------
    # CHANGE 5: Process each source separately
    # ---------------------------------------

    for doc in documents:

        doc_source = doc.metadata.get(
            "source_file",
            doc.metadata.get("source", "unknown")
        )

        # Do not mix content from different documents
        if current_source != doc_source:

            flush_buffer()

            current_source = doc_source

            hierarchy = {
                "chapter_number": None,
                "chapter_title": None,
                "section_number": None,
                "section_title": None,
                "subsection_number": None,
                "subsection_title": None,
                "heading": None
            }

            source_metadata = {
                "source_file": str(doc_source),
                "file_type": str(
                    doc.metadata.get("file_type", "unknown")
                )
            }

        page = doc.metadata.get("page")

        page_number = (
            page + 1
            if isinstance(page, int)
            else None
        )

        for line in doc.page_content.splitlines():

            line = line.strip()

            if not line:
                continue

            # ---------------------------------------
            # Detect Markdown headings
            # ---------------------------------------

            md_match = markdown_heading.match(line)

            # ---------------------------------------
            # Detect chapter headings
            # ---------------------------------------

            chapter_match = chapter_heading.match(line)

            # ---------------------------------------
            # Detect numbered headings
            # ---------------------------------------

            number_match = numbered_heading.match(line)

            if md_match:

                flush_buffer()

                level = len(md_match.group(1))
                heading = md_match.group(2)

                if level == 1:
                    update_hierarchy(
                        "chapter", None, heading
                    )

                elif level == 2:
                    update_hierarchy(
                        "section", None, heading
                    )

                elif level == 3:
                    update_hierarchy(
                        "subsection", None, heading
                    )

                else:
                    update_hierarchy(
                        "heading", None, heading
                    )

            elif chapter_match:

                flush_buffer()

                number = chapter_match.group(2)
                title = chapter_match.group(3) or line

                update_hierarchy(
                    "chapter", number, title
                )

            elif number_match:

                flush_buffer()

                number = number_match.group(1).rstrip(".")
                title = number_match.group(2)

                depth = number.count(".") + 1

                if depth == 1:
                    update_hierarchy(
                        "section", number, title
                    )

                elif depth == 2:
                    update_hierarchy(
                        "subsection", number, title
                    )

                else:
                    update_hierarchy(
                        "heading", number, title
                    )

            else:

                buffer.append(line)

                # Track pages containing actual chunk content
                if page_number is not None:
                    page_numbers.add(page_number)

    # ---------------------------------------
    # Process the final remaining buffer
    # ---------------------------------------

    flush_buffer()

    print(
        f"Input documents: {len(documents)}\n"
        f"Output chunks: {len(structured_chunks)}"
    )

    if structured_chunks:

        print("\nExample chunk:")
        print(structured_chunks[0].page_content[:300])

        print("\nMetadata:")
        print(structured_chunks[0].metadata)

    return structured_chunks


In [5]:
chunks=split_documents(all_documents)
chunks

Input documents: 116
Output chunks: 386

Example chunk:
1
Supplemental Materials
Machine learning links unresolving secondary pneumonia to mortality in patients with severe
pneumonia, including COVID-19
Catherine A. Gao1 *, Nikolay S. Markov1 *, Thomas Stoeger2*, Anna Pawlowski3, Mengjia Kang1, Prasanth
Nannapaneni3, Rogan A. Grant1, Chiagozie Pickens1 ,

Metadata:
{'source_file': '16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf', 'file_type': 'pdf', 'pages': '1', 'chunk_index': 0}


[Document(metadata={'source_file': '16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf', 'file_type': 'pdf', 'pages': '1', 'chunk_index': 0}, page_content='1\nSupplemental Materials\nMachine learning links unresolving secondary pneumonia to mortality in patients with severe\npneumonia, including COVID-19\nCatherine A. Gao1 *, Nikolay S. Markov1 *, Thomas Stoeger2*, Anna Pawlowski3, Mengjia Kang1, Prasanth\nNannapaneni3, Rogan A. Grant1, Chiagozie Pickens1 , James M. Walter1 , Jacqueline M. Kruser1,4, Luke\nRasmussen5, Daniel Schneider3, Justin Starren5, Helen K. Donnelly1, Alvaro Donayre1, Yuan Luo5, GR Scott\nBudinger1,6 **, Richard G. Wunderink1,6 **, Alexander V. Misharin1,6 **, Benjamin D. Singer1,6 **,\nThe NU SCRIPT Study Investigators\n*co-first authors\n**co-senior authors'),
 Document(metadata={'source_file': '16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf', 'file_type': 'pdf', 'section_number': '1', 'section_title': 'Division of Pulmonary and Critical Care Medicine, Department of Medicine, N

### embedding And vectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully. Embedding dimension: 384


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_13940\540119965.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [8]:
import os
import hashlib
import chromadb
import numpy as np

from typing import List, Any

In [18]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store."""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = r"E:\RAG\RAG-test\data\vector_store"
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory

        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection."""

        try:
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                }
            )

            print(
                f"Vector store initialized. Collection: "
                f"{self.collection_name}"
            )

            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    @staticmethod
    def generate_document_id(doc, chunk_index):
        """
        Generate a deterministic, source-aware document ID.

        The same source, page, chunk index and content
        will always produce the same ID.
        """

        # 1. Get source filename
        source = (
            doc.metadata.get("source_file")
            or doc.metadata.get("source")
            or "unknown_source"
        )

        source = str(source).replace("\\", "/").strip().lower()

        # 2. Get page information
        page = (
            doc.metadata.get("page_label")
            or doc.metadata.get("page")
            or "unknown_page"
        )

        page = str(page).strip()

        # 3. Normalize chunk content
        content = " ".join(doc.page_content.split())

        # 4. Generate content hash
        content_hash = hashlib.sha256(
            content.encode("utf-8")
        ).hexdigest()

        # 5. Construct unique key
        unique_key = (
            f"{source}|{page}|{chunk_index}|{content_hash}"
        )

        # 6. Generate final ID
        return hashlib.sha256(
            unique_key.encode("utf-8")
        ).hexdigest()

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """Add or update documents without creating duplicate IDs."""

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        if len(documents) == 0:
            print("No documents to add.")
            return

        print(f"Processing {len(documents)} documents...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            # Generate deterministic ID
            doc_id = self.generate_document_id(doc, i)
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)

            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            metadatas.append(metadata)

            # Store original document content
            documents_text.append(doc.page_content)

            # Convert embedding to list
            embeddings_list.append(
                np.asarray(embedding).tolist()
            )

        # Check for duplicate IDs within current batch
        if len(ids) != len(set(ids)):
            raise ValueError(
                "Duplicate document IDs found in the current batch. "
                "Check source metadata, page numbers and chunk indexing."
            )

        try:
            # Insert new records or update existing records
            self.collection.upsert(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Successfully processed {len(documents)} documents"
            )

            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error adding documents: {e}")
            raise


In [19]:
chunks

[Document(metadata={'source_file': '16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf', 'file_type': 'pdf', 'pages': '1', 'chunk_index': 0}, page_content='1\nSupplemental Materials\nMachine learning links unresolving secondary pneumonia to mortality in patients with severe\npneumonia, including COVID-19\nCatherine A. Gao1 *, Nikolay S. Markov1 *, Thomas Stoeger2*, Anna Pawlowski3, Mengjia Kang1, Prasanth\nNannapaneni3, Rogan A. Grant1, Chiagozie Pickens1 , James M. Walter1 , Jacqueline M. Kruser1,4, Luke\nRasmussen5, Daniel Schneider3, Justin Starren5, Helen K. Donnelly1, Alvaro Donayre1, Yuan Luo5, GR Scott\nBudinger1,6 **, Richard G. Wunderink1,6 **, Alexander V. Misharin1,6 **, Benjamin D. Singer1,6 **,\nThe NU SCRIPT Study Investigators\n*co-first authors\n**co-senior authors'),
 Document(metadata={'source_file': '16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf', 'file_type': 'pdf', 'section_number': '1', 'section_title': 'Division of Pulmonary and Critical Care Medicine, Department of Medicine, N

In [20]:
vectorstore = VectorStore()

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 2034


In [21]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store in the vector database
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 386 texts...


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Generated embeddings with shape: (386, 384)
Processing 386 documents...
Successfully processed 386 documents
Total documents in collection: 2420


### Retriever Pipeline From VectorStore

In [23]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(
        self,
        vector_store: vectorstore,
        embedding_manager: EmbeddingManager
    ):

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:

        print("\n" + "=" * 70)
        print("USER QUESTION")
        print("=" * 70)
        print(query)

        print("\nRetrieval Settings:")
        print(f"Top K: {top_k}")
        print(f"Score Threshold: {score_threshold}")

        # --------------------------------
        # STEP 1: Generate query embedding
        # --------------------------------

        query_embedding = (
            self.embedding_manager.generate_embeddings([query])[0]
        )

        # --------------------------------
        # STEP 2: Search ChromaDB
        # --------------------------------

        try:

            results = self.vector_store.collection.query(
                query_embeddings=[
                    query_embedding.tolist()
                ],
                n_results=top_k,
                include=[
                    "documents",
                    "metadatas",
                    "distances"
                ]
            )

            retrieved_docs = []

            if results["documents"] and results["documents"][0]:

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"]

                # --------------------------------
                # STEP 3: Process retrieved results
                # --------------------------------

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids[0], documents, metadatas, distances)
                ):

                    # For cosine distance:
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:

                        retrieved_docs.append({

                            # 1. Original user question
                            "query": query,

                            # 2. Retrieved text
                            "content": document,

                            # 3. Source metadata
                            "metadata": metadata,

                            # 4. Retrieval information
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "id": doc_id,
                            "rank": i + 1

                        })

            # --------------------------------
            # STEP 4: Display results
            # --------------------------------

            if retrieved_docs:

                print(
                    f"\nRetrieved {len(retrieved_docs)} "
                    "relevant chunks"
                )

                for result in retrieved_docs:

                    print("\n" + "=" * 70)

                    print("QUERY:")
                    print(result["query"])

                    print("\nRETRIEVED CONTENT:")
                    print(result["content"])

                    print("\nSOURCE METADATA:")

                    for key, value in result["metadata"].items():
                        print(f"{key}: {value}")

                    print("\nRELEVANCE SCORE:")
                    print(round(result["similarity_score"], 4))

                    print("RANK:", result["rank"])

                    print("=" * 70)

            else:
                print("\nNo relevant documents found.")

            return retrieved_docs

        except Exception as e:

            print(f"Error during retrieval: {e}")
            return []


# Initialize retriever
rag_retriever = RAGRetriever(
    vectorstore,
    embedding_manager
)


In [24]:
rag_retriever

In [25]:
rag_retriever.retrieve("Ground-based observational data")


USER QUESTION
Ground-based observational data

Retrieval Settings:
Top K: 5
Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)

No relevant documents found.


[]

In [26]:
rag_retriever.retrieve("Integrated local indicators of spatial connectivity")



USER QUESTION
Integrated local indicators of spatial connectivity

Retrieval Settings:
Top K: 5
Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)

Retrieved 5 relevant chunks

QUERY:
Integrated local indicators of spatial connectivity

RETRIEVED CONTENT:
Global and local indicators of spatial connectivity for
areal data
Álvaro Briz-Redón 1* and Marco Ruiz-Valderrama 2
1Departament d’Estadística i Investigació Operativa, Univ ersitat de
València, València, Spain
2Facultat de Ciències Matemàtiques, Universitat de Valènci a, València,
Spain
*Corresponding author ( alvaro.briz@uv.es)
September 11, 2026
Abstract
Methods for areal data commonly describe spatial structure through global au-
tocorrelation measures or local indicators based on neighb oring values. In many
applications, however, it is also relevant to know whether h igh- or low-valued areas
form connected spatial structures. Motivated by topologic al data analysis, we de-
velop global and local indicators of spatial connectivity f or observations deﬁned on
an areal adjacency graph.
We construct thresholded spatial graphs and use t

[{'query': 'Integrated local indicators of spatial connectivity',
  'content': 'Global and local indicators of spatial connectivity for\nareal data\nÁlvaro Briz-Redón 1* and Marco Ruiz-Valderrama 2\n1Departament d’Estadística i Investigació Operativa, Univ ersitat de\nValència, València, Spain\n2Facultat de Ciències Matemàtiques, Universitat de Valènci a, València,\nSpain\n*Corresponding author ( alvaro.briz@uv.es)\nSeptember 11, 2026\nAbstract\nMethods for areal data commonly describe spatial structure through global au-\ntocorrelation measures or local indicators based on neighb oring values. In many\napplications, however, it is also relevant to know whether h igh- or low-valued areas\nform connected spatial structures. Motivated by topologic al data analysis, we de-\nvelop global and local indicators of spatial connectivity f or observations deﬁned on\nan areal adjacency graph.\nWe construct thresholded spatial graphs and use the Betti-0 c urve to record their\nnumber of connected 

In [27]:
from collections import defaultdict

results = vectorstore.collection.get(
    include=["documents", "metadatas"]
)

content_groups = defaultdict(list)

for doc_id, document, metadata in zip(
    results["ids"],
    results["documents"],
    results["metadatas"]
):
    content_groups[document].append({
        "id": doc_id,
        "source": metadata.get("source_file"),
        "page": metadata.get("page_label"),
        "doc_index": metadata.get("doc_index")
    })

# Print first 5 duplicate groups
count = 0

for content, records in content_groups.items():

    if len(records) > 1:
        print("\nDuplicate content:")
        print(content[:200])

        for record in records:
            print(record)

        count += 1

        if count == 5:
            break


Duplicate content:
1 
Supplemental Materials 
 
Machine learning links unresolving secondary pneumonia to mortality in patients with severe 
pneumonia, including COVID-19 
 
Catherine A. Gao1 *, Nikolay S. Markov1 *, Th
{'id': 'doc_3aebbd7c_0', 'source': '16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf', 'page': '1', 'doc_index': 0}
{'id': 'doc_0ea518bc_0', 'source': '16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf', 'page': '1', 'doc_index': 0}
{'id': 'doc_71d063e7_0', 'source': '16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf', 'page': '1', 'doc_index': 0}

Duplicate content:
School of Medicine, Chicago, IL, USA 
2 Department of Chemical and Biological Engineering, Northwestern University, McCormick School of 
Engineering, Evanston, IL, USA 
3 Northwestern Medicine Enterpr
{'id': 'doc_d92baba5_1', 'source': '16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf', 'page': '1', 'doc_index': 1}
{'id': 'doc_badea987_1', 'source': '16fb6046-8d12-4fd4-9b97-0377075b1e30.pdf', 'page': '1', 'doc_index': 1}
{'id': 'doc_bedba3

### RAG Pipeline- VectorDB To LLM Output Generation

In [39]:
import os
from dotenv import load_dotenv
load_dotenv()

#print(os.getenv("GROQ_API_KEY"))

True

In [40]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [41]:
class GroqLLM:
    def __init__(self, model_name: str ="openai/gpt-oss-120b", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    


In [42]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: openai/gpt-oss-120b
Groq LLM initialized successfully!


In [43]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Areal data on a spatial graph")


USER QUESTION
Areal data on a spatial graph

Retrieval Settings:
Top K: 5
Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)

Retrieved 5 relevant chunks

QUERY:
Areal data on a spatial graph

RETRIEVED CONTENT:
2026).
In the context of areal data, observations are attached to a ﬁ xed set of spatial units,
with adjacency encoded by a spatial graph. In this setting, t hresholding the observations
produces induced subgraphs whose organization changes wit h the threshold. Active areas
may remain spatially separated or become connected as less e xtreme values are included.
The zeroth Betti number, or Betti-0, summarizes this evolut ion by counting the connected
components of each thresholded graph. Our framework uses th is quantity to describe how
high- or low-valued regions connect across thresholds.
In particular, in this paper, we propose global and local ind icators of spatial con-
nectivity for areal data. The superlevel analysis examines the spatial organization of
high-valued areas by retaining, at each threshold, the area s whose values are at or above
that thres

[{'query': 'Areal data on a spatial graph',
  'content': '2026).\nIn the context of areal data, observations are attached to a ﬁ xed set of spatial units,\nwith adjacency encoded by a spatial graph. In this setting, t hresholding the observations\nproduces induced subgraphs whose organization changes wit h the threshold. Active areas\nmay remain spatially separated or become connected as less e xtreme values are included.\nThe zeroth Betti number, or Betti-0, summarizes this evolut ion by counting the connected\ncomponents of each thresholded graph. Our framework uses th is quantity to describe how\nhigh- or low-valued regions connect across thresholds.\nIn particular, in this paper, we propose global and local ind icators of spatial con-\nnectivity for areal data. The superlevel analysis examines the spatial organization of\nhigh-valued areas by retaining, at each threshold, the area s whose values are at or above\nthat threshold together with the adjacency links between th em. Conver

### Integration Vectordb Context pipeline With LLM output

In [44]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [45]:
answer=rag_simple("What is Areal data on a spatial graph?",rag_retriever,llm)
print(answer)


USER QUESTION
What is Areal data on a spatial graph?

Retrieval Settings:
Top K: 3
Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)

Retrieved 3 relevant chunks

QUERY:
What is Areal data on a spatial graph?

RETRIEVED CONTENT:
2026).
In the context of areal data, observations are attached to a ﬁ xed set of spatial units,
with adjacency encoded by a spatial graph. In this setting, t hresholding the observations
produces induced subgraphs whose organization changes wit h the threshold. Active areas
may remain spatially separated or become connected as less e xtreme values are included.
The zeroth Betti number, or Betti-0, summarizes this evolut ion by counting the connected
components of each thresholded graph. Our framework uses th is quantity to describe how
high- or low-valued regions connect across thresholds.
In particular, in this paper, we propose global and local ind icators of spatial con-
nectivity for areal data. The superlevel analysis examines the spatial organization of
high-valued areas by retaining, at each threshold, the area s whose values are at or above
t

### Enhanced RAG Pipeline Features

In [46]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("An object in the Einstein desert", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


USER QUESTION
An object in the Einstein desert

Retrieval Settings:
Top K: 3
Score Threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)

Retrieved 3 relevant chunks

QUERY:
An object in the Einstein desert

RETRIEVED CONTENT:
also infer the relative lens-source trigonometric parallax,𝜋 rel =0.202 +0.054
−0.052 mas. The direction,
spectral type, and flux of the source star indicate that it is located in the Milky Way’s central bulge.
Adopting a bulge distance of𝐷𝑠=7.93+0.54
−0.51 kpc for the line of sight toward this event (24), the lens
distance is𝐷 𝑙 =3.05+0.58
−0.43 kpc.
An object in the Einstein desert
With𝜃 E≈18.6µas, KMT-2024-BLG-0792/OGLE-2024-BLG-0516 falls within the Einstein desert
identified by previous studies (Fig. 3) (12, 11). In principle, without the lens mass measurement,
this event could have been classified as either an FFP event (like previous events below the desert,
𝜃E ≲9µas) or a BD event (like previous events above the desert,𝜃 E ≳25µas).
Various scenarios could explain the paucity of objects in the Einstein desert, which have masses
statistically inferre

In [47]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is Areal data on a spatial graph?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])


USER QUESTION
what is Areal data on a spatial graph?

Retrieval Settings:
Top K: 3
Score Threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)

Retrieved 3 relevant chunks

QUERY:
what is Areal data on a spatial graph?

RETRIEVED CONTENT:
2026).
In the context of areal data, observations are attached to a ﬁ xed set of spatial units,
with adjacency encoded by a spatial graph. In this setting, t hresholding the observations
produces induced subgraphs whose organization changes wit h the threshold. Active areas
may remain spatially separated or become connected as less e xtreme values are included.
The zeroth Betti number, or Betti-0, summarizes this evolut ion by counting the connected
components of each thresholded graph. Our framework uses th is quantity to describe how
high- or low-valued regions connect across thresholds.
In particular, in this paper, we propose global and local ind icators of spatial con-
nectivity for areal data. The superlevel analysis examines the spatial organization of
high-valued areas by retaining, at each threshold, the area s whose values are at or above
t